*This code is a companion to the book **Mastering PyTorch and Lightning: A Step-by-Step Practical Guide with QA** by Aghiles Kebaili*
> This notebook contains the raw code for Chapter 11: Mastering the Lightning Trainer. It shows how a minimal smoke test grows into a full training configuration with hardware, precision, and monitored checkpoints.

Once the model and data plumbing are in place, the Trainer becomes the orchestrator: it manages fitting, validation, prediction, and checkpointed execution without moving the model logic itself.

## Automating Training and Evaluation

### Step 0: Improting Previously Implemented Classes

In [ ]:
import os
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import pytorch_lightning as pl

# This setup reuses the DataModule and model from earlier chapters, then lets the Trainer handle execution.
class VisionDataModule(pl.LightningDataModule):
    def __init__(
        self,
        data_dir: str = "./data",
        batch_size: int = 256,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = min(4, os.cpu_count() or 1)
        self.pin_memory = torch.cuda.is_available()

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(
                (0.5, 0.5, 0.5),
                (0.5, 0.5, 0.5),
            ),
        ])

    def prepare_data(self):
        datasets.CIFAR10(
            self.data_dir, train=True, download=True
        )
        datasets.CIFAR10(
            self.data_dir, train=False, download=True
        )

    def setup(self, stage: str | None = None):
        if stage in ("fit", "validate", None):
            if not hasattr(self, "cifar_train"):
                full_dataset = datasets.CIFAR10(
                    self.data_dir,
                    train=True,
                    transform=self.transform,
                )
                generator = torch.Generator().manual_seed(42)
                self.cifar_train, self.cifar_val = random_split(
                    full_dataset,
                    [45000, 5000],
                    generator=generator,
                )

        if stage in ("test", "predict", None):
            if not hasattr(self, "cifar_test"):
                self.cifar_test = datasets.CIFAR10(
                    self.data_dir,
                    train=False,
                    transform=self.transform,
                )

    def _make_loader(self, dataset, shuffle):
        return DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=shuffle,
            num_workers=self.num_workers,
            pin_memory=self.pin_memory,
            persistent_workers=self.num_workers > 0,
        )

    def train_dataloader(self):
        return self._make_loader(self.cifar_train, shuffle=True)

    def val_dataloader(self):
        return self._make_loader(self.cifar_val, shuffle=False)

    def test_dataloader(self):
        return self._make_loader(self.cifar_test, shuffle=False)

    def predict_dataloader(self):
        return self._make_loader(self.cifar_test, shuffle=False)
    

class LitCIFARClassifier(pl.LightningModule):
    def __init__(self, lr: float = 1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        return self.backbone(x)

    def _shared_step(self, batch, prefix):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        self.log(
            f"{prefix}_loss",
            loss,
            on_epoch=True,
            prog_bar=True,
            sync_dist=prefix != "train",
            batch_size=x.size(0),
        )
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        self._shared_step(batch, "test")

    def predict_step(self, batch, batch_idx):
        x, _ = batch
        return self(x).argmax(dim=1)

    def configure_optimizers(self):
        return torch.optim.AdamW(
            self.parameters(), lr=self.hparams.lr
        )

### Step 1: Verify Integration with a Smoke Run

In [2]:
# Before launching a long training, run a quick smoke test to verify that all components
# (model, DataModule, optimizer, backward pass) work together without errors.
import pytorch_lightning as pl

model = LitCIFARClassifier(lr=1e-3)
datamodule = VisionDataModule(
    data_dir="./data",
    batch_size=64,
)

# fast_dev_run=True runs exactly one batch through train, val, and test.
# This quickly exposes integration issues without requiring a full epoch.
smoke_trainer = pl.Trainer(
    fast_dev_run=True,
    accelerator="auto",
    devices=1,
    logger=False,
    enable_checkpointing=False,
)

smoke_trainer.fit(model, datamodule=datamodule)

# After one complete gradient step, we confirm the loop executed successfully.
assert smoke_trainer.global_step == 1


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.
You are using a CUDA device ('NVIDIA GeForce RTX 3060') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type       | Params | Mode 
------------------------------------------------
0 | backbone | Sequential | 1.2 K  | train
------------------------------------------------
1.2 K     Trainable params
0         Non-trainable params
1.2 K     Total params
0.005     Total estimated model params size (MB)
6         Modules in train mode
0         M

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1` reached.


### Step 2: Distinguish the Managed Stages

## Hardware & Precision Configuration

### Step 1: Build a Portable Execution Policy

In [ ]:
# Hardware choices belong in the Trainer, not in the model. This keeps the LightningModule
# portable across CPU, GPU, and other accelerators without code changes.
import torch

use_cuda = torch.cuda.is_available()
# Mixed precision reduces activation memory and can improve throughput on CUDA hardware.
# FP32 remains the safe baseline on CPU or unsupported systems.
precision = "16-mixed" if use_cuda else "32-true"

# Pack Trainer settings together so they can be reviewed, logged, and modified as a unit.
trainer_policy = {
    "accelerator": "auto",        # Automatically selects from available backends (gpu, mps, cpu)
    "devices": "auto",             # Lets Lightning choose how many devices to use
    "strategy": "auto",             # Selects the distributed strategy if needed
    "precision": precision,         # Numeric precision for arithmetic and memory efficiency
}

assert trainer_policy["precision"] in {
    "16-mixed", "32-true"
}


## Extending the Trainer with Callbacks

### Step 1: Define Stopping and Persistence Policies

In [ ]:
# Callbacks separate operational concerns (stopping, saving, monitoring) from the model logic.
# Both depend on the same monitored metric name (e.g., "val_loss").
from pytorch_lightning.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
)

# Early stopping tracks validation loss. If it does not improve by min_delta within patience checks,
# training stops gracefully. This saves compute time and prevents overfitting.
early_stop = EarlyStopping(
    monitor="val_loss",           # Watch this metric
    mode="min",                   # Lower is better
    patience=3,                   # Stop after 3 checks without sufficient improvement
    min_delta=0.0,                # Minimum change to count as improvement
)

# ModelCheckpoint saves the model when the monitored metric reaches a new best value.
# save_top_k=1 keeps only the best checkpoint to manage disk space.
# save_last=True also preserves the latest state for resuming interrupted runs.
checkpoint = ModelCheckpoint(
    dirpath="./saved_models",
    filename="best-{epoch:02d}-{val_loss:.3f}",  # Embed epoch and metric in filename
    monitor="val_loss",
    mode="min",
    save_top_k=1,                 # Keep only the single best checkpoint
    save_last=True,               # Also save the latest state for resumption
)


### Step 2: Assemble the Full Training Configuration

In [5]:
# This configuration assembles all tested components: model, data, hardware settings, and callbacks.
# The effective global batch size is local_batch * num_devices * accumulation_steps.
model = LitCIFARClassifier(lr=1e-3)
datamodule = VisionDataModule(
    data_dir="./data",
    batch_size=512,  # Local batch size per device
)

trainer = pl.Trainer(
    max_epochs=100,
    accelerator=trainer_policy["accelerator"],
    devices=trainer_policy["devices"],
    strategy=trainer_policy["strategy"],
    precision=trainer_policy["precision"],
    callbacks=[early_stop, checkpoint],  # Attach callbacks for stopping and persistence
    gradient_clip_val=1.0,                # Clip gradients to prevent instability
    accumulate_grad_batches=2,            # Simulate a larger batch by accumulating 2 micro-batches
    log_every_n_steps=10,                 # Log metrics every 10 steps
)

# The fit() method launches the training loop with all managed stages: train, validate, and early stop.
trainer.fit(model, datamodule=datamodule)

# After training, the checkpoint object holds paths to the best model found by validation.
assert checkpoint.best_model_path


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type       | Params | Mode 
------------------------------------------------
0 | backbone | Sequential | 1.2 K  | train
------------------------------------------------
1.2 K     Trainable params
0         Non-trainable params
1.2 K     Total params
0.005     Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.


### Step 4: Evaluate and Predict from the Selected Checkpoint

In [6]:
test_results = trainer.test(
    model=model,
    datamodule=datamodule,
    ckpt_path="best",
)

predictions = trainer.predict(
    model=model,
    datamodule=datamodule,
    ckpt_path="best",
)

assert isinstance(test_results, list)
assert isinstance(predictions, list)

Restoring states from the checkpoint path at G:\work\ebooks\mastering-pytorch-lightning-book\pytorch-core-foundation\saved_models\best-epoch=99-val_loss=1.690.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at G:\work\ebooks\mastering-pytorch-lightning-book\pytorch-core-foundation\saved_models\best-epoch=99-val_loss=1.690.ckpt


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │      1.6820148229599      │
└───────────────────────────┴───────────────────────────┘

Restoring states from the checkpoint path at G:\work\ebooks\mastering-pytorch-lightning-book\pytorch-core-foundation\saved_models\best-epoch=99-val_loss=1.690.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at G:\work\ebooks\mastering-pytorch-lightning-book\pytorch-core-foundation\saved_models\best-epoch=99-val_loss=1.690.ckpt


Predicting: |          | 0/? [00:00<?, ?it/s]